In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge

## Подключение корня проекта

Ноутбук находится в папке `notebooks/`, а код проекта — в `src/`.

Чтобы импортировать функции из `src`, добавляем корень проекта в `sys.path`. Это удобнее, чем дублировать код загрузки данных в каждом ноутбуке.

In [ ]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data import load_train_test

from src.utils import (
    evaluate_model,
    create_submission,
    save_experiment_result,
)

In [ ]:
train, test = load_train_test()

## Разделение на признаки и целевую переменную

`SalePrice` — это целевая переменная, которую нужно предсказывать.

`Id` не несёт полезной информации для модели: это технический идентификатор строки, поэтому удаляем его из признаков.

Цель преобразуем через `np.log1p`, потому что метрика Kaggle считается по логарифмам цен. Также логарифмирование делает распределение цены более спокойным: дорогие дома меньше доминируют в ошибке.

In [37]:
y = np.log1p(train["SalePrice"])

X = train.drop(columns=["SalePrice", "Id"])

## Preprocessing pipeline

На baseline-этапе preprocessing намеренно простой.

Для числовых признаков:
1. Заполняет пропуски медианой.

2. Приводит числовые признаки к сопоставимому масштабу.

Для категориальных признаков:

1. Заполняет пропуски самым частым значением.

2. One-Hot Encoding.

In [38]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## Baseline-модель: Ridge Regression

В качестве первой модели используем `Ridge`.

Ridge -- это линейная регрессия с L2-регуляризацией. Она хорошо подходит для baseline, потому что:

- быстро обучается;
- даёт понятную стартовую точку качества;
- лучше обычной линейной регрессии работает при большом количестве one-hot encoded признаков;
- регуляризация помогает ограничить слишком большие коэффициенты и снизить риск переобучения.

In [39]:
model = Ridge(alpha=10)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

## Cross-validation

Для оценки качества используем 5-fold cross-validation.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_rmse, cv_std, scores = evaluate_model(
    pipeline,
    X,
    y,
    cv,
)

print(f"CV RMSE log mean: {cv_rmse:.5f}")
print(f"CV RMSE log std:  {cv_std:.5f}")

CV RMSE log mean: 0.14679
CV RMSE log std:  0.03932


* Разброс качества между фолдами довольно высокий. Возможная причина -- выбросы, найденные на этапе EDA: при KFold они могут неравномерно попадать в разные фолды и заметно влиять на RMSE. На следующих этапах проверю это после обработки выбросов.

In [41]:
pipeline.fit(X, y)
X_test = test.drop(columns=["Id"])

preds_log = pipeline.predict(X_test)
preds = np.expm1(preds_log)

## Сохранение результата эксперимента

Сохраняем результат baseline в таблицу экспериментов.

In [42]:
submission = pd.DataFrame(
    {
        "Id": test["Id"],
        "SalePrice": preds,
    }
)

submission.head()

,Id,SalePrice
0,1461,114305.333169
1,1462,145492.635478
2,1463,170682.737994
3,1464,192627.874861
4,1465,198253.498839


In [43]:
submission.to_csv(
    PROJECT_ROOT / "submissions" / "submission_baseline.csv",
    index=False,
)

## Результаты

- Средний CV RMSE (log): ~0.14
- Результат на Kaggle: **0.13375**

---

## Наблюдения

- Pipeline работает корректно и без утечек данных
- Даже простая линейная модель показывает адекватный результат
- Основные зависимости в данных (площадь, качество, гараж) уже улавливаются
- Обработка пропусков крайне примитивная (без учёта смысла признаков)

---

## Ограничения текущего решения

- Пропуски обрабатываются универсально 
- Не добавлены новые признаки
- Не удалены выбросы
- Используется только одна модель